# Reproducing Ramachandran et al. (2025) from dandiset 000945

**Paper.** Ramachandran S, Gao H, Yttri E, Yu K, He B (2025). Parameter-dependent cell-type
specific effects of transcranial focused ultrasound stimulation in an awake head-fixed rodent
model. *J Neural Eng* 22(2) 026022.
[doi:10.1088/1741-2552/adbb1f](https://doi.org/10.1088/1741-2552/adbb1f). The claims file was
written from the version of record as served by Europe PMC (PMC11920894): main text, Methods,
figure legends and Discussion; the supplementary materials were not supplied, and no claim rests
on them.

**Dandiset.** [DANDI:000945](https://dandiarchive.org/dandiset/000945/0.250213.2212), version
0.250213.2212. This is the version the paper cites (reference 50,
doi:10.48324/dandi.000945/0.250213.2212). The four later published versions (0.250711.1746,
0.250829.0425, 0.250829.1250) and the draft hold the same 75 files with the same blobs and
sizes, as the archive's asset lists show, so the choice does not change the data analysed.

**What the dandiset holds.** Sorted units (spike times; a cell-type label in 60 of 75 files) and
the trial table of each recording. There are no waveforms, no raw or filtered voltage, no
electrode positions and no sham recordings. The condition of each recording is coded only in the
file's `identifier` (`<rat>_<PRF>_<TBD>[_2][_anes]`).

| Claim | Tested here |
|---|---|
| fig3-cell-type-sorting | Counts tested with the dataset's labels; the k-means features cannot be computed (no waveforms) |
| fig4-rsu-time-locked-both-states | Blocked: the criterion compares with sham, and there is no sham |
| fig4hi-fsu-delayed-latency | Tested |
| fig4h-awake-latency-wider | Tested |
| fig5a-rsu-time-locked-prf-awake | Tested, without the sham parts |
| fig5c-rsu-time-locked-prf-anesth | Tested (two labelled rats), without the sham parts |
| rsu-time-locked-anesthesia-factor | Tested |
| fig5bd-fsu-time-locked-no-prf | Tested |
| fig5e-rsu-delayed-prf-awake | Tested, without the sham part |
| fig5f-fsu-delayed-prf-awake | Tested, without the sham part |
| fig5gh-delayed-prf-anesth | Tested |
| delayed-anesthesia-factor | Tested |
| fig6ad-time-adjacent-batch | Blocked: the second batch of five rats is not in the dandiset |
| fig6ef-rsu-ispta-rise-then-fall | Tested |
| fig6fh-fsu-ispta | Tested |
| fig5a-1500-highest-time-locked-mean | Tested |
| fig7a-rsu-time-locked-dc | Tested, without the sham part |
| fig7a-60-highest-time-locked-mean | Tested |
| fig7b-fsu-time-locked-dc | Tested |
| fig7c-rsu-delayed-dc | Tested, without the sham part |
| fig7d-fsu-delayed-dc | Tested, without the sham part |
| sham-no-auditory-response | Blocked: no sham recordings, and no test named |

In [ ]:
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols

import tfus

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = tfus.asset_list()
sessions = tfus.load_all(assets)
print(f"loaded {len(sessions)} files from dandiset {tfus.DANDISET_ID} version {tfus.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
inv = pd.DataFrame([{
    "path": s["path"], "subject_id": s["subject_id"], "identifier": s["identifier"], "rat": s["rat"],
    "state": s["state"], "prf": s["prf"], "tbd": s["tbd"], "repeat": s["repeat"],
    "description": s["session_description"], "age": s["age"], "start": s["session_start"][:16],
    "units": s["n_units"], "labelled": int((~np.isnan(s["label"])).sum()), "trials": s["n_trials"],
    "units_cols": " ".join(s["units_colnames"]), "trial_cols": " ".join(s["trials_colnames"]),
    "acq_proc": " ".join(s["acquisition"] + s["processing"]) or "-"} for s in sessions])
print("Files:", len(inv), "; subject folders:", inv.subject_id.nunique(), sorted(inv.subject_id.unique()))
print("Rats named in identifiers:", sorted(inv.rat.unique()))
print("Files whose subject_id differs from the rat in the identifier:")
print(inv[inv.subject_id != inv.rat][["path", "identifier"]].to_string(index=False))
print("\nUnits table columns:", inv.units_cols.value_counts().to_dict())
print("Trials table columns:", inv.trial_cols.value_counts().to_dict())
print("Acquisition or processing modules:", inv.acq_proc.value_counts().to_dict())
print("Ages:", inv.age.value_counts().to_dict())
print("Session descriptions by state in the identifier:")
print(pd.crosstab(inv.description, inv.state).to_string())
print("\nTrials per file:", inv.trials.value_counts().to_dict())
print(inv[inv.trials != 500][["identifier", "trials"]].to_string(index=False))
print("\nLabels: files with celltype_label", int((inv.labelled > 0).sum()), "of", len(inv),
      "; units in files without it:", int(inv[inv.labelled == 0].units.sum()))
print("Every identifier parses as <rat>_<PRF>_<TBD>[_2][_anes|_anes2]: True (parse_identifier raised no error)")

There are no waveforms, voltage traces, electrode positions or processing modules: units hold
only spike times and, in 60 files, a label ("1 = RSU, 2 = FSU"). Every file says age P24W (about
5.5 months), where the Methods give 7, 9 and 10 months for the three series. The 15 files without
labels are all anesthetized recordings, and they are the only files whose description says
"Awake Model Anes"; the labelled anesthetized copies (subject folders BH497 and BH514) are
described as "Awake S1 Stimulation". The identifiers of those copies name rats BH498 and BH512.
Five files hold fewer than the 500 trials the Methods describe.

In [ ]:
dup = tfus.find_duplicates(sessions)
print("Groups of files with identical trial times:")
print(dup[dup.group_size > 1].sort_values(["duplicate_of", "path"], na_position="first")
      [["path", "identifier", "kept", "duplicate_of"]].to_string(index=False))
by_path = {s["path"]: s for s in sessions}
rows = []
for r in dup[~dup.kept].itertuples():
    a, b = by_path[r.path], by_path[r.duplicate_of]
    same_counts = a["n_units"] == b["n_units"] and np.array_equal(a["n_spikes"], b["n_spikes"])
    same_first = np.array_equal(a["first_spikes"], b["first_spikes"])
    rows.append({"file": r.identifier + " (" + r.subject_id + ")", "kept": b["identifier"] + " (" + b["subject_id"] + ")",
                 "units": f'{a["n_units"]}/{b["n_units"]}', "spikes": f'{a["n_spikes"].sum()}/{b["n_spikes"].sum()}',
                 "same spike count per unit": same_counts, "same first 5000 spike times": same_first})
print(pd.DataFrame(rows).to_string(index=False))


def shared_spikes(p1, p2):
    out = []
    for p in (p1, p2):
        url = next(a["url"] for a in assets if a["path"] == p)
        with h5py.File(remfile.File(url), "r") as f:
            out.append(np.unique(np.round(f["units/spike_times"][:], 5)))
    return len(out[0]), len(out[1]), len(np.intersect1d(out[0], out[1]))


n1, n2, n12 = shared_spikes("sub-BH498/sub-BH498_ses-20240306T175805_ecephys.nwb",
                            "sub-BH508/sub-BH508_ses-20240307T002354_ecephys.nwb")
print(f"\nBH498_30_200 and BH508_30_200_2: distinct spike times {n1} and {n2}, shared to 10 us: {n12}")
kept_paths = set(dup[dup.kept].path)
kept = [s for s in sessions if s["path"] in kept_paths]
print("Recordings counted once:", len(kept), "of", len(sessions), "files")

Twelve files are second copies. The five anesthetized BH498 recordings are each deposited twice,
under BH498 without labels and under BH497 with labels; the BH512 anesthetized recordings at
300, 1500, 3000 and 4500 Hz are deposited under BH512 without labels and under BH514 with labels,
and `BH512_1500_200_anes2` is a third copy of the 1500 Hz recording with the same trial times and
a different sort (21,329 fewer spikes). In each pair the trial times, the number of spikes of
every unit and the first spike times are identical. The labelled copy is kept, and the rat is
taken from the identifier (BH498, BH512), which the unlabelled copies' subject folders confirm.
`BH498_30_200` and `BH508_30_200_2` are filed under two rats but share all 203 trial times and
18,860 spike times; they are one recording sorted twice, and only `BH498_30_200` is kept (the
other is a `_2` repeat). 64 recordings remain.

In [ ]:
units = tfus.unit_table(kept)
cond = units.groupby(["state", "prf", "tbd", "repeat", "rat"]).session.first().reset_index()
print("Recordings kept, by state, PRF, TBD and rat (x = one recording; R = a `_2` repeat):")
cond["mark"] = np.where(cond.repeat, "R", "x")
print(cond.pivot_table(index=["state", "prf", "tbd"], columns="rat", values="mark",
                       aggfunc=lambda v: "+".join(sorted(v))).fillna("").to_string())
iti = np.concatenate([np.diff(s["trial_times"]) for s in kept])
tl = np.concatenate([s["trial_len"] for s in kept])
print(f"\nInter-trial interval: {iti.min():.3f} to {iti.max():.3f} s, median {np.median(iti):.3f} "
      f"(Methods: 2.5 s with 10% jitter); {int((iti > 2.75).sum())} intervals above 2.75 s")
print(f"stop_time - start_time: {tl.min():.3f} to {tl.max():.3f} s")
cover = [(s["identifier"], s["trial_times"][0], s["trial_times"][-1], *s["spike_range"]) for s in kept]
bad = [c for c in cover if c[1] - 0.5 < c[3] - 1 or c[2] + 1.5 > c[4] + 1]
print("Recordings whose spikes do not span the trials from -0.5 to +1.5 s (1 s slack):", bad or "none")
print("Spike times sorted within every unit:", all(s["spikes_sorted"] for s in kept))

The dandiset holds the awake PRF series of seven rats (BH498, BH506, BH508, BH512, BH519, BH526,
BH527), ten `_2` repeats of awake PRF conditions in BH498, BH506 and BH508, the awake 6% and 60%
DC conditions (TBD 40 and 400 us) of three rats (BH506, BH521, BH526), anesthetized PRF series
for BH498 (all five PRFs), BH512 (no 30 Hz) and BH521 (300, 3000 and 4500 Hz, unlabelled), and
anesthetized 6% and 60% DC recordings of BH512 (unlabelled). The paper describes ten rats in the
first batch, five more in a second batch with a sham condition, and a DC series with its own 30%
recordings at 10 months; none of the second batch, no sham and no separate 30% DC recording are
here. Intervals between trials fit the Methods' 2.5 s with 10% jitter, apart from one long gap.
The trial table's `stop_time` is always 2.2 s after `start_time`; the Methods give no 2.2 s
period, and `start_time` is taken as the trigger, which the next check supports.

In [ ]:
lab_units = units[units.celltype.notna()]
print("Units in kept recordings:", len(units), "; labelled:", len(lab_units),
      "; unlabelled (BH512 DC and BH521 anesthetized):", int(units.celltype.isna().sum()))
print("Units with no baseline spike (left out):", int((lab_units.base_hz == 0).sum()))
print("\nBaseline rate (Hz) by label, over units: median, IQR")
print(lab_units.groupby("celltype").base_hz.describe(percentiles=[.25, .5, .75])[["count", "25%", "50%", "75%", "max"]].round(2).to_string())
per_ses = lab_units.groupby(["session", "celltype"]).base_hz.median().unstack()
print(f"Recordings where the FSU median rate exceeds the RSU median: {(per_ses.FSU > per_ses.RSU).sum()} of {len(per_ses.dropna())}")
print("\nLabelled units per state and type (kept recordings):")
print(pd.crosstab([lab_units.state, lab_units.tbd], lab_units.celltype).to_string())

The labels split the units about evenly. FSU, the putative fast-spiking interneurons, are
expected to fire faster; the comparison of baseline rates above says whether the labels follow
that (it is a consistency check, not a test of the labels).

In [ ]:
# Clock check: the population spike count of each recording in 0-67 ms after each trial start
# against the 0.5 s before it, at the true trial times and 1.25 s later (mid-interval).
rows = []
for s in kept:
    c = s["counts"].sum(axis=0)        # trials x windows, summed over units
    i = {k: tfus.window_index(k) for k in ("base", "tl_w1", "shift_base", "shift_tl")}
    true = c[:, i["tl_w1"]] / 0.067 - c[:, i["base"]] / 0.5
    shift = c[:, i["shift_tl"]] / 0.067 - c[:, i["shift_base"]] / 0.5
    rows.append({"session": s["identifier"], "state": s["state"], "prf": s["prf"], "tbd": s["tbd"],
                 "t_true": stats.ttest_1samp(true, 0).statistic, "t_shifted": stats.ttest_1samp(shift, 0).statistic})
clock = pd.DataFrame(rows)
print(clock.round(2).sort_values(["state", "tbd", "prf"]).to_string(index=False))
print(f"\nRecordings with t > 3 at the true times: {(clock.t_true > 3).sum()} of {len(clock)}; "
      f"at shifted times: {(clock.t_shifted > 3).sum()}")
print(f"Median t: true {clock.t_true.median():.2f}, shifted {clock.t_shifted.median():.2f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
t = tfus.FINE_EDGES[:-1] + tfus.FINE / 2
for ax, (st, prf, tbd) in zip(axes, [("awake", 1500, 400), ("awake", 3000, 200), ("anesthetized", 1500, 200)]):
    for code, name in [(1.0, "RSU"), (2.0, "FSU")]:
        prof = []
        for s in kept:
            if (s["state"], s["prf"], s["tbd"]) != (st, prf, tbd):
                continue
            p = s["psth5"][s["label"] == code]
            b = p[:, :100].mean(1)
            prof.append(p[b > 0] / b[b > 0, None])
        m = 100 * np.concatenate(prof).mean(0)
        seg = {f"{a:+.3f} to {b:+.3f} s": m[(t > a) & (t < b)].mean() for a, b in ((-0.075, 0), (0, 0.075), (0.075, 0.15))}
        print(f"{st} {prf} Hz {tbd} us {name}: mean % of baseline", {k: round(float(v), 1) for k, v in seg.items()},
              f"; peak {m.max():.0f}% at {t[np.argmax(m)]:.4f} s")
        ax.plot(t, m, label=f"{name} (n={len(np.concatenate(prof))}, range {m.min():.0f}-{m.max():.0f}%)")
    ax.axvspan(0, 0.067, color="0.85")
    ax.set_title(f"{st} {prf} Hz, TBD {tbd} us")
    ax.set_xlabel("time from trial start (s)")
    ax.legend(fontsize=7)
axes[0].set_ylabel("rate, % of baseline (5 ms bins)")
plt.tight_layout()
plt.show()

Ten of the 64 recordings show a population rise in the 67 ms after `start_time` (t > 3) and none
shows one at the shifted times. In the conditions plotted, the mean rate is near baseline in the
75 ms before `start_time` (99.8 to 103.6%), rises in the 75 ms after it (103.7 to 196.9%), the
span of the 67 ms ultrasound, and falls back in the next 75 ms (100.2 to 115.1%). So
`start_time` is the trigger, and spikes and trials share a clock in those recordings, which
include awake and anesthetized ones and four of the eight rats (BH498, BH506, BH512, BH521). In
the other 54 recordings the check cannot tell a shared clock from no response.

## Readings and selections used in every claim

The claims file's header fixes the method; the dataset decides which of its readings can run.

- **Cell type.** K0: the dataset carries labels, so they are used. K1 and K2 (k-means on the
  initial and after-hyperpolarization phase durations) cannot run: the files hold no waveforms.
- **Trials.** The files hold 500 trials, the 505 delivered less the first five (T1, Methods 2.4);
  T2 (all 505) cannot run. Five recordings hold 203 to 339 trials and are used as they are.
- **Exclusion.** E1 = E2: the dataset marks no excluded unit or interval.
- **Windows.** W1 (exact window) and W2 (overlapping PSTH bins) are both run.
- **Groups.** G1 only: there are no sham recordings, so G2 cannot run.
- **Repeats** (not named by the claims file; the dandiset adds them). The `_2` recordings repeat
  a condition in the same rat. The Methods describe one recording per parameter set per rat
  ("Recordings for a single rat for each set of parameters were completed on the same day",
  Methods 2.3) and count "the total number of neurons in each group" (Methods 2.6). Rep1 counts
  every recording once, repeats included; Rep2 leaves the repeats out. Both are run.
- **Anesthetized data.** Only two rats' anesthetized recordings carry labels (BH498 at all five
  PRFs, BH512 without 30 Hz); BH521's three are unlabelled and cannot enter a cell-type test.
- **Sham.** The dandiset holds no sham recording. Parts of a criterion that compare with sham
  are reported as not tested and the claim is judged on its other parts, as the claims file's
  header says for an anesthetized sham; the same is done for the awake sham.

Where readings give different outcomes, the reading best supported by the paper is adopted as an
assumption, and every reading is still shown: W2, as the Methods derive responses from the binned
PSTH ("resulting in a histogram of average spikes per time bin for each neuron", Methods 2.4);
Rep2, as the Methods describe one recording per parameter set per rat (Methods 2.3); and, for
the duty cycle claims, C1 (defined there), as the paper's DC series compares the levels within
the same rats.

In [ ]:
L = units[units.celltype.notna() & (units.base_hz > 0)].copy()
PRFS = [30, 300, 1500, 3000, 4500]
READINGS = [(w, r) for w in ("W1", "W2") for r in ("Rep1", "Rep2")]


def col(kind, state, w):
    if kind == "tl":
        return "tl_w1" if w == "W1" else "tl_w2"
    return tfus.delayed_window(state, w)


def select(state, ct, rep, tbd=200, prf=None):
    d = L[(L.state == state) & (L.celltype == ct) & (L.tbd == tbd)]
    if prf is not None:
        d = d[d.prf == prf]
    if rep == "Rep2":
        d = d[~d.repeat]
    return d


def prf_test(state, ct, kind, w, rep):
    d = select(state, ct, rep)
    c = col(kind, state, w)
    return tfus.group_test({p: d[d.prf == p][c].values for p in PRFS})


def sig(post, a, b):
    return bool(tfus.pair(post, a, b).p_bonf < 0.05)


def lower(post, a, b):
    """a significantly below b."""
    r = tfus.pair(post, a, b)
    return bool(r.p_bonf < 0.05 and r.mean_a < r.mean_b)


def show(o, post, label):
    print(f"{label}: Kruskal-Wallis H = {o['H']:.1f}, p = {o['p']:.2g}; n = {o['n']}")
    print("  means:", {k: round(v, 1) for k, v in o["mean"].items()},
          " medians:", {k: round(v, 1) for k, v in o["median"].items()})
    print(post[["a", "b", "mean_a", "mean_b", "p", "p_bonf", "sig"]].round(4).to_string(index=False))


def by_rat(state, ct, kind, w, rep, tbd=200):
    d = select(state, ct, rep, tbd=tbd)
    c = col(kind, state, w)
    return d.pivot_table(index="rat", columns="prf" if tbd == 200 else "dc", values=c, aggfunc="mean").round(1)


print("Labelled units with a baseline, by state and type:", L.groupby(["state", "celltype"]).size().to_dict())

## fig3-cell-type-sorting

> Units are sorted into fast spiking units (FSU) and regular spiking units (RSU) by k-means
> clustering of the initial phase and after-hyperpolarization phase durations of their action
> potentials; in one animal's recordings this gave 28 FSU and 32 RSU, 60 in total.

In [ ]:
cnt = units[units.celltype.notna()].groupby(["rat", "session", "state"]).celltype.value_counts().unstack(fill_value=0)
cnt["total"] = cnt.sum(axis=1)
ok = lambda c: (abs(c.FSU - 28) <= 2.8) & (abs(c.RSU - 32) <= 3.2) & (abs(c.total - 60) <= 6)
print("R1, one recording: recordings with all three counts in tolerance (FSU 28 +- 2.8, RSU 32 +- 3.2, total 60 +- 6)")
print(cnt[ok(cnt)].to_string())
print(f"({len(cnt)} labelled recordings; totals range {cnt.total.min()} to {cnt.total.max()}, FSU share "
      f"{(cnt.FSU / cnt.total).min():.2f} to {(cnt.FSU / cnt.total).max():.2f})")
lu = units[units.celltype.notna()].copy()
lu["series"] = np.where(lu.tbd == 200, lu.state + " PRF", lu.state + " DC")
cnt2 = lu.groupby(["rat", "series"]).celltype.value_counts().unstack(fill_value=0)
cnt2["total"] = cnt2.sum(axis=1)
print("\nR2, all of one rat's recordings in one series:")
print(cnt2.to_string())
print("R2 matches:", cnt2[ok(cnt2)].index.tolist() or "none")
print("\nThe clusters' separation on the initial phase and after-hyperpolarization durations cannot be "
      "computed: the files hold no waveforms (units columns:", sorted(set(inv.units_cols)), ")")

**Result: partially reproduced.** With the dataset's labels (K0), two single recordings give all
three counts within tolerance (R1): BH527_30_200 with 28 FSU, 34 RSU and 62 units, and
BH508_4500_200_2 with 28 FSU, 30 RSU and 58 units. No rat's series matches (R2; totals 77 to 616).
The labels follow the expected rate difference in 46 of 59 recordings (FSU median baseline rate
above the RSU median; 8.60 against 5.84 Hz over units). The k-means sorting itself, and the
separation of the clusters on the initial phase and after-hyperpolarization durations, cannot be
tested: the files hold no waveforms. Only the count part of the criterion could be tested, and it
holds.

## fig4-rsu-time-locked-both-states (blocked)

> RSUs show time-locked responses to tFUS, an increase in firing rate during the 67 ms of
> stimulation, in both awake and anesthetized recordings.

The criterion compares the 1500 Hz, 200 us RSU responses with the sham sessions of each state.
The dandiset holds no sham recording, so the claim is blocked. The responses the criterion would
have compared are shown, without a test.

In [ ]:
for st in ("awake", "anesthetized"):
    for w in ("W1", "W2"):
        for rep in ("Rep1", "Rep2"):
            d = select(st, "RSU", rep, prf=1500)[col("tl", st, w)]
            print(f"{st:12s} {w} {rep}: RSU at 1500 Hz, n = {len(d)}, mean {d.mean():.1f}%, median {d.median():.1f}%, "
                  f"units above 100%: {(d > 100).sum()}")

## fig4hi-fsu-delayed-latency

> FSU delayed responses to tFUS at 1500 Hz PRF begin later in awake rats (mean onset 0.51 s,
> spread 0.14 s) than under anesthesia (0.24 s, spread 0.08 s), and the onset distributions
> differ significantly by a Wilcoxon rank-sum test.

Sessions: FSU of the 1500 Hz, 200 us recordings. Q1 and Q2 select the same recordings here: the
awake 30% DC condition has no recording of its own (it is the 1500 Hz, 200 us recording), and
batch 2 is not in the dandiset. Every combination of S1/S2, D1/D2, L1/L2 and Rep1/Rep2 is run.

In [ ]:
lat_rows = []
for s in kept:
    if s["prf"] != 1500 or s["tbd"] != 200:
        continue
    for u in range(s["n_units"]):
        if s["label"][u] != 2.0:
            continue
        r = {"session": s["identifier"], "rat": s["rat"], "state": s["state"], "repeat": s["repeat"]}
        for sem in ("S1", "S2"):
            for first in ("D1", "D2"):
                for timing in ("L1", "L2"):
                    r[f"{sem}{first}{timing}"] = tfus.latency(s["psth"][u], s["counts"][u, :, 0], sem, first, timing)
        lat_rows.append(r)
lat = pd.DataFrame(lat_rows)
combos = [c for c in lat.columns if c.startswith("S")]
res = []
for c in combos:
    for rep in ("Rep1", "Rep2"):
        d = lat if rep == "Rep1" else lat[~lat.repeat]
        aw, an = d[d.state == "awake"][c].dropna(), d[d.state == "anesthetized"][c].dropna()
        n_aw, n_an = (d.state == "awake").sum(), (d.state == "anesthetized").sum()
        p = stats.ranksums(aw, an).pvalue
        res.append({"reading": c + " " + rep, "awake n": f"{len(aw)}/{n_aw}", "anes n": f"{len(an)}/{n_an}",
                    "awake mean": aw.mean(), "awake SD": aw.std(), "anes mean": an.mean(), "anes SD": an.std(),
                    "p": p, "anes in tol": abs(an.mean() - 0.24) <= 0.08, "awake in tol": abs(aw.mean() - 0.51) <= 0.14,
                    "awake later, p<0.001": bool(p < 0.001 and aw.median() > an.median())})
lat_res = pd.DataFrame(res)
print("FSU at 1500 Hz, 200 us; n = units with a significant delayed response / FSU")
print(lat_res.round(4).to_string(index=False))
main = lat[lat.state == "awake"]
print("\nBy rat, S1D1L1 Rep1: units with a delayed response and their mean latency (s)")
print(lat.groupby(["state", "rat"]).S1D1L1.agg(["size", "count", "mean"]).round(3).to_string())
for st in ("awake", "anesthetized"):
    d = lat[lat.state == st]
    v = d.S1D1L1.dropna()
    print(f"{st}: latencies range {v.min():.3f} to {v.max():.3f} s; first allowed bin (D1) holds {np.mean(v < 0.08):.2f} of them")

**Result: not reproduced.** Under none of the 16 readings do awake FSU onsets come later than
anesthetized ones at p < 0.001: rank-sum p ranges from 0.018 to 0.85, and in 6 readings the
anesthetized mean is the later one. The awake means (0.40 to 0.58 s) are within 0.51 +- 0.14 s, but
the anesthetized means (0.38 to 0.52 s) are all outside 0.24 +- 0.08 s. Most FSU qualify as having a
delayed response (193 to 265 of 306 awake under Rep1, 34 or 39 of 50 anesthetized). The anesthetized FSU come
from two rats.

## fig4h-awake-latency-wider

> Awake FSU delayed-response onset latencies have a wider spread than anesthetized ones and appear
> to have two peaks.

In [ ]:
print(lat_res[["reading", "awake SD", "anes SD"]].assign(wider=lambda d: d["awake SD"] > d["anes SD"]).round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
bins = tfus.EDGES[tfus.N_BASE_BINS:]
for st in ("anesthetized", "awake"):
    v = lat[lat.state == st].S1D1L1.dropna()
    h, _ = np.histogram(v, bins)
    peaks = [round(float(bins[i]), 3) for i in range(len(h)) if h[i] > 0 and (i == 0 or h[i] > h[i - 1]) and (i == len(h) - 1 or h[i] >= h[i + 1])]
    print(f"{st}: histogram (35.7 ms bins from 0 s) counts {h.tolist()}; local maxima at {peaks} s")
    ax.hist(v, bins, alpha=0.6, label=f"{st} (n={len(v)}, {v.min():.2f}-{v.max():.2f} s)")
ax.set_xlabel("onset latency (s), S1 D1 L1 Rep1")
ax.set_ylabel("FSU")
ax.legend(fontsize=8)
plt.show()

**Result: not reproduced.** The awake SD exceeds the anesthetized SD in 10 of 16 readings, and
never by more than 0.02 s (0.381 to 0.393 s awake, 0.367 to 0.406 s anesthetized); the criterion
asks for every reading. Under S1 D1 L1 Rep1 the awake histogram has its largest count in the first
allowed bin (61 of 262 at 0.071 s) and several smaller local maxima; the anesthetized histogram
(39 units) is flat with many single-count maxima.

## fig5a-rsu-time-locked-prf-awake

> In awake rats PRF is a significant factor for RSU time-locked responses; 30 Hz and 300 Hz PRF
> give lower responses than every higher PRF (so 30 Hz is also below 300 Hz), 1500, 3000 and
> 4500 Hz do not differ, and sham differs from every PRF but 30 Hz.

Parts (5) and (6) compare with sham and cannot be tested; the claim is judged on parts (1) to (4).

In [ ]:
rows = []
for w, rep in READINGS:
    o, post = prf_test("awake", "RSU", "tl", w, rep)
    if (w, rep) == ("W1", "Rep1"):
        show(o, post, "awake RSU 0-67 ms, W1 Rep1")
    rows.append({"reading": f"{w} {rep}", "(1) KW": o["p"] < 0.05,
                 "(2) 30 below each": all(lower(post, 30, b) for b in (300, 1500, 3000, 4500)),
                 "(3) 300 below each": all(lower(post, 300, b) for b in (1500, 3000, 4500)),
                 "(4) no pair of 1500/3000/4500": not any(sig(post, a, b) for a, b in ((1500, 3000), (1500, 4500), (3000, 4500))),
                 "highest mean": max(o["mean"], key=o["mean"].get)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMean response by rat (W1 Rep1):")
print(by_rat("awake", "RSU", "tl", "W1", "Rep1").to_string())

**Result: not reproduced.** PRF is a significant factor (Kruskal-Wallis p = 3.3e-16, W1 Rep1),
but the post hoc pattern differs. The highest mean is at 3000 Hz (108.2%; 1500 Hz 98.5%), and
3000 Hz differs from 1500 and 4500 Hz (corrected p < 1e-4), so part (4) fails under every reading.
300 Hz is not below 1500 or 4500 Hz (part 3 fails under every reading), and 30 Hz is below every
higher PRF only under W2 Rep1 (part 2). Parts (5) and (6) need the sham and are not tested. Means
by rat vary widely (BH526 at 64 to 101%, BH506 at 100 to 130%).

## fig5c-rsu-time-locked-prf-anesth

> Under anesthesia RSU time-locked responses depend on PRF as in the awake model, the only
> difference being that 30 Hz and 300 Hz do not differ significantly.

Two labelled rats: BH498 (all five PRFs) and BH512 (no 30 Hz). The repeats reading does not apply
(no anesthetized repeats remain). Parts (5) and (6) compare with sham and cannot be tested.

In [ ]:
rows = []
for w in ("W1", "W2"):
    o, post = prf_test("anesthetized", "RSU", "tl", w, "Rep1")
    show(o, post, f"anesthetized RSU 0-67 ms, {w}")
    rows.append({"reading": w, "(1) KW": o["p"] < 0.05,
                 "(2) 30 below 1500/3000/4500": all(lower(post, 30, b) for b in (1500, 3000, 4500)),
                 "(3) 300 below each": all(lower(post, 300, b) for b in (1500, 3000, 4500)),
                 "(4) no pair of 1500/3000/4500": not any(sig(post, a, b) for a, b in ((1500, 3000), (1500, 4500), (3000, 4500))),
                 "(7) 30 = 300": not sig(post, 30, 300)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMean response by rat (W1):")
print(by_rat("anesthetized", "RSU", "tl", "W1", "Rep1").to_string())

**Result: not reproduced.** Under anesthesia (two labelled rats) the Kruskal-Wallis is
significant (p = 7.3e-08 W1, 7.7e-07 W2) and 30 Hz and 300 Hz do not differ (part 7 holds), but the
other parts do not: 1500 Hz is the peak (107.0%), 30 Hz is not below 3000 or 4500 Hz, 300 Hz is
not below 3000 or 4500 Hz, and 1500 Hz differs from 4500 Hz (and from 3000 Hz under W1). Only part
(1) and part (7) of the testable parts hold. The 1500 Hz effect comes from BH512 (110.4% against
101.7% in BH498).

## rsu-time-locked-anesthesia-factor

> Anesthesia is a significant factor in the RSU time-locked response by two-way ANOVA, and it
> lowers the average magnitude of RSU responses.

In [ ]:
def anova(ct, kind, w, rep, model):
    d = pd.concat([select(st, ct, rep) for st in ("awake", "anesthetized")])
    d = d.assign(y=[r[col(kind, r.state, w)] for _, r in d.iterrows()])
    form = "y ~ C(state) * C(prf)" if model == "A1" else "y ~ C(state) + C(prf)"
    a = sm.stats.anova_lm(ols(form, d).fit(), typ=2)
    return {"reading": f"{model} {w} {rep}", "n": len(d), "F state": a.loc["C(state)", "F"], "p state": a.loc["C(state)", "PR(>F)"],
            "awake mean": d[d.state == "awake"].y.mean(), "anes mean": d[d.state == "anesthetized"].y.mean()}


def anova_table(ct, kind):
    return pd.DataFrame([anova(ct, kind, w, rep, m) for m in ("A1", "A2") for w, rep in READINGS])

In [ ]:
t_rsu = anova_table("RSU", "tl")
print(t_rsu.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nPer-PRF means (W1 Rep1):")
print(pd.concat({st: select(st, "RSU", "Rep1").groupby("prf").tl_w1.mean() for st in ("awake", "anesthetized")}, axis=1).round(1).to_string())
print("\nVariant, the two rats with labelled anesthetized data only (BH498, BH512), W1 Rep1, A1:")
d = pd.concat([select(st, "RSU", "Rep1") for st in ("awake", "anesthetized")])
d = d[d.rat.isin(["BH498", "BH512"])].assign(y=lambda x: x.tl_w1)
a = sm.stats.anova_lm(ols("y ~ C(state) * C(prf)", d).fit(), typ=2)
print(f"  F = {a.loc['C(state)', 'F']:.2f}, p = {a.loc['C(state)', 'PR(>F)']:.3g}; awake mean {d[d.state == 'awake'].y.mean():.1f}, "
      f"anesthetized {d[d.state == 'anesthetized'].y.mean():.1f}")

**Result: not reproduced.** Anesthesia is a significant factor (p = 0.011 to 0.018 over the eight
readings, all above 0.01), and the direction is the reverse of the claim: the anesthetized mean
(102.6 to 102.9%) is above the awake mean (100.1 to 100.4%). With only the two rats that have
labelled anesthetized data, the effect is not significant (p = 0.425).

## fig5bd-fsu-time-locked-no-prf

> FSU time-locked responses show no significant differences across PRF levels, in either the
> awake or the anesthetized model.

A claim of absence, judged on the Kruskal-Wallis across the five PRFs (G1) at 0.05 and the post
hoc tests at Bonferroni-corrected 0.05.

In [ ]:
rows = []
for st in ("awake", "anesthetized"):
    for w, rep in READINGS:
        if st == "anesthetized" and rep == "Rep2":
            continue
        o, post = prf_test(st, "FSU", "tl", w, rep)
        if w == "W1" and rep == "Rep1":
            show(o, post, f"{st} FSU 0-67 ms, W1 Rep1")
        med = o["median"]
        rows.append({"state": st, "reading": f"{w} {rep}", "KW p": o["p"], "post hoc pairs significant": int(post.sig.sum()),
                     "median range": f"{min(med.values()):.1f}-{max(med.values()):.1f}",
                     "mean range": f"{min(o['mean'].values()):.1f}-{max(o['mean'].values()):.1f}"})
print(pd.DataFrame(rows).round(6).to_string(index=False))
print("\nMean response by rat, awake (W1 Rep1):")
print(by_rat("awake", "FSU", "tl", "W1", "Rep1").to_string())

**Result: not reproduced.** By the paper's test FSU time-locked responses do differ across PRF:
Kruskal-Wallis p = 1.8e-12 awake and 5.5e-04 anesthetized (W1 Rep1), with 5 or 6 significant post
hoc pairs awake and 2 or 3 anesthetized under every reading. The effects are small: group medians
span 98.7 to 103.4% awake (W1 Rep1) and 99.8 to 103.9% anesthetized; the awake 3000 Hz mean is 106.8%.

## fig5e-rsu-delayed-prf-awake

> Awake RSU delayed responses (0.5 to 1 s) depend on PRF, are highest at 1500 Hz and 3000 Hz with
> lower responses at both lower and higher PRFs, and every PRF is significantly higher than sham.

Part (3) compares with sham and cannot be tested. The 0.5 to 1 s window is on bin edges, so W1
and W2 are the same.

In [ ]:
rows = []
for rep in ("Rep1", "Rep2"):
    o, post = prf_test("awake", "RSU", "del", "W1", rep)
    show(o, post, f"awake RSU 0.5-1 s, {rep}")
    m = o["mean"]
    top2 = sorted(m, key=m.get)[-2:]
    rows.append({"reading": rep, "(1) KW": o["p"] < 0.05,
                 "(2) R1: 1500 and 3000 above 30, 300, 4500": all(lower(post, a, b) for b in (1500, 3000) for a in (30, 300, 4500)),
                 "(2) R2: top two means are 1500 and 3000": sorted(top2) == [1500, 3000],
                 "(2) R2: both above 30 and 4500": all(lower(post, a, b) for b in (1500, 3000) for a in (30, 4500))})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMean response by rat (Rep1):")
print(by_rat("awake", "RSU", "del", "W1", "Rep1").to_string())

**Result: not reproduced.** PRF is a significant factor (p = 6e-20 Rep1, 4.5e-15 Rep2), but
the two highest means are at 1500 Hz and 30 Hz (106.7 and 106.6% Rep1; 108.8 and 106.6% Rep2),
not 1500 and 3000 Hz: 3000 Hz (102.3%) is below 1500 Hz (corrected p = 0.0053) and not above 30
Hz. Part (2) fails under both of its readings; part (3) needs the sham and is not tested.

## fig5f-fsu-delayed-prf-awake

> Awake FSU delayed responses depend on PRF, are strongest at 4500 Hz, differ between 30 Hz and
> the three highest PRFs and between 300 Hz and the two highest, and at 30 Hz and 300 Hz are not
> significantly higher than sham.

Part (5) compares with sham and cannot be tested.

In [ ]:
rows = []
for rep in ("Rep1", "Rep2"):
    o, post = prf_test("awake", "FSU", "del", "W1", rep)
    show(o, post, f"awake FSU 0.5-1 s, {rep}")
    rows.append({"reading": rep, "(1) KW": o["p"] < 0.05, "(2) 4500 highest": max(o["mean"], key=o["mean"].get) == 4500,
                 "(3) 30 differs from 1500/3000/4500": all(sig(post, 30, b) for b in (1500, 3000, 4500)),
                 "(4) 300 differs from 3000/4500": all(sig(post, 300, b) for b in (3000, 4500))})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMean response by rat (Rep1):")
print(by_rat("awake", "FSU", "del", "W1", "Rep1").to_string())

**Result: not reproduced.** PRF is a significant factor (p = 4.6e-12 Rep1) and 30 Hz differs
from 1500, 3000 and 4500 Hz (part 3), but the strongest mean is at 30 Hz (103.5%), not 4500 Hz
(100.3%), and 300 Hz does not differ from 4500 Hz (corrected p = 1.0). Two of the three testable
parts after (1) fail under both readings; part (5) needs the sham and is not tested.

## fig5gh-delayed-prf-anesth

> Under anesthesia PRF is a significant factor for the delayed responses (0.2 to 0.7 s) of both
> RSUs and FSUs, which broadly prefer higher PRFs.

In [ ]:
rows = []
for ct in ("RSU", "FSU"):
    for w in ("W1", "W2"):
        o, post = prf_test("anesthetized", ct, "del", w, "Rep1")
        d = select("anesthetized", ct, "Rep1")[[col("del", "anesthetized", w), "prf"]]
        hi = d[d.prf >= 1500].iloc[:, 0].mean()
        lo = d[d.prf <= 300].iloc[:, 0].mean()
        rows.append({"type": ct, "reading": w, "KW H": o["H"], "KW p": o["p"], "n": o["n"],
                     "mean 1500-4500": hi, "mean 30-300": lo, "higher PRFs higher": hi > lo})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nMeans by rat and PRF, RSU W1:")
print(by_rat("anesthetized", "RSU", "del", "W1", "Rep1").to_string())
print("FSU W1:")
print(by_rat("anesthetized", "FSU", "del", "W1", "Rep1").to_string())

**Result: not reproduced.** Under anesthesia PRF is not a significant factor for delayed
responses: Kruskal-Wallis p = 0.16 (W1) and 0.12 (W2) for RSU, 0.16 and 0.18 for FSU. The pooled
1500 to 4500 Hz means are above the 30 and 300 Hz means by less than one percentage point (RSU
99.97 against 99.15%, FSU 99.87 against 99.35%, W1). The data are two rats.

## delayed-anesthesia-factor

> Anesthesia is a significant factor in the delayed response for both RSUs and FSUs by two-way
> ANOVA, and it lowers the average response magnitude.

In [ ]:
tabs = {ct: anova_table(ct, "del") for ct in ("RSU", "FSU")}
for ct, t in tabs.items():
    print(ct)
    print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
chk = pd.DataFrame({"reading": tabs["RSU"].reading,
                    "RSU p<0.001": tabs["RSU"]["p state"] < 0.001, "FSU p<0.01": tabs["FSU"]["p state"] < 0.01,
                    "both p<0.05": (tabs["RSU"]["p state"] < 0.05) & (tabs["FSU"]["p state"] < 0.05),
                    "anes lower, both types": (tabs["RSU"]["anes mean"] < tabs["RSU"]["awake mean"]) & (tabs["FSU"]["anes mean"] < tabs["FSU"]["awake mean"])})
print(chk.to_string(index=False))
print("\nFSU per-rat means, awake recordings with and without the repeats (0.5-1 s):")
print(select("awake", "FSU", "Rep1").groupby(["rat", "repeat"]).del_awake.agg(["size", "mean"]).round(1).to_string())

**Result: reproduced, resting on an assumption.** Without the `_2` repeats (Rep2, the reading
adopted; see the readings above and the summary), anesthesia is a significant factor for both
types under all four model and window readings: RSU p < 1e-8 (F = 34.4 to 34.9) and FSU p = 0.0013
to 0.0015, with the anesthetized mean lower in both (RSU 99.7 against 104.6%, FSU 99.7 against
101.3%). With the repeats (Rep1) the RSU part holds (F = 21.3 to 21.6, p < 1e-5) but the FSU effect
is p = 0.038 to 0.041, which misses the 0.01 bound: partially reproduced under that reading.

## fig6ad-time-adjacent-batch (blocked)

> In the five rats recorded awake and anesthetized in the same week, time-locked responses to
> the low PRF and 1500 Hz differ significantly in both states, with a reduced response under
> anesthesia.

The ten rats in the dandiset are the first batch's (each has awake PRF recordings), and no file
is from a sham or a two-PRF protocol; the five rats of the second batch are not in it. Blocked as
data missing; no section is run.

## fig6ef-rsu-ispta-rise-then-fall

> Awake RSU mean responses, time-locked and delayed, rise with ISPTA up to around 60 mW/cm2 and
> then decline.

In [ ]:
def ispta_table(ct):
    rows = []
    for w, rep in READINGS:
        for kind in ("tl", "del"):
            if kind == "del" and w == "W2":
                continue
            d = select("awake", ct, rep)
            c = col(kind, "awake", w)
            g = d.groupby("prf")[c]
            m, se = g.mean(), g.sem()
            rows.append({"window": kind, "reading": f"{w} {rep}",
                         **{f"{p} Hz ({tfus.PRF_ISPTA[p]})": f"{m[p]:.1f} +- {se[p]:.1f}" for p in PRFS},
                         "_m": m})
    return rows


rows = ispta_table("RSU")
out = []
for r in rows:
    m = r.pop("_m")
    peak = m.idxmax()
    rise = all(m[PRFS[i]] <= m[PRFS[i + 1]] for i in range(PRFS.index(peak))) if peak in (1500, 3000) else False
    out.append({**r, "peak": peak, "peak at 1500/3000": peak in (1500, 3000), "rises 30 to peak": rise,
                "4500 below peak": m[4500] < m[peak]})
print("Mean +- SEM (%) by PRF (ISPTA mW/cm2):")
print(pd.DataFrame(out).to_string(index=False))

**Result: partially reproduced, resting on an assumption.** Time-locked RSU means rise from 30 Hz
(96.8%) to a peak at 3000 Hz (60.26 mW/cm2; 108.2%) and fall at 4500 Hz (99.2%) under W1 Rep1, W2
Rep1 and W2 Rep2; under W1 Rep2 the 300 Hz mean (98.2%) is 0.1 point above the 1500 Hz mean
(98.1%), against SEMs of 1.1 and 1.3. The delayed means do not rise to their peak: 106.6% at 30 Hz,
101.2% at 300 Hz and 106.7% at 1500 Hz (Rep1). The time-locked window holds under W2, the reading
adopted (responses come from the binned PSTH), and the delayed window does not.

## fig6fh-fsu-ispta

> Awake FSUs show no time-locked response, and their delayed response increases almost linearly
> with ISPTA.

The time-locked part takes the awake outcome of fig5bd-fsu-time-locked-no-prf.

In [ ]:
rows = []
for rep in ("Rep1", "Rep2"):
    m = select("awake", "FSU", rep).groupby("prf").del_awake.mean()
    x = [tfus.PRF_ISPTA[p] for p in PRFS]
    y = [m[p] for p in PRFS]
    rows.append({"reading": rep, **{f"{p} Hz": round(m[p], 1) for p in PRFS},
                 "Spearman rho": stats.spearmanr(x, y)[0], "Pearson r": stats.pearsonr(x, y)[0]})
print(pd.DataFrame(rows).round(3).to_string(index=False))
o, post = prf_test("awake", "FSU", "tl", "W1", "Rep1")
print(f"\nTime-locked part (from fig5bd, awake, W1 Rep1): Kruskal-Wallis p = {o['p']:.2g}, "
      f"{int(post.sig.sum())} significant post hoc pairs")

**Result: not reproduced.** The time-locked part fails (fig5bd: awake Kruskal-Wallis p =
1.8e-12). The delayed FSU means do not increase with ISPTA: 103.5, 98.7, 101.8, 100.6 and 100.3%
from 30 to 4500 Hz (Spearman rho = -0.4, Pearson r = -0.27, Rep1; rho = -0.3 Rep2).

## fig5a-1500-highest-time-locked-mean

> Among PRFs, 1500 Hz gives the highest mean RSU time-locked response in awake rats, although not
> significantly above 3000 and 4500 Hz.

In [ ]:
rows = []
for w, rep in READINGS:
    o, post = prf_test("awake", "RSU", "tl", w, rep)
    rows.append({"reading": f"{w} {rep}", **{f"{p} Hz": round(o["mean"][p], 1) for p in PRFS},
                 "highest": max(o["mean"], key=o["mean"].get),
                 "p_bonf 1500 vs 3000": tfus.pair(post, 1500, 3000).p_bonf, "p_bonf 1500 vs 4500": tfus.pair(post, 1500, 4500).p_bonf})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** The highest mean RSU time-locked response is at 3000 Hz under every
reading (107.7 to 108.7%), and it is significantly above 1500 Hz (corrected p = 0.0040 or less);
1500 Hz gives 98.1 to 98.7%.

## Duty cycle claims (fig7a to fig7d)

The DC series is 1500 Hz with TBD 40, 200 and 400 us (6, 30 and 60%). The dandiset holds 6% and
60% recordings for three rats (BH506, BH521, BH526) and no 30% recording of its own: the 30%
condition can only be the 1500 Hz, 200 us recordings of the PRF series. Two readings are run
(the claims file does not name them; they come from what the dandiset holds): C1, the 30%
recordings of the same rats (BH506 and BH526; BH521 has none), and C2, every awake 1500 Hz,
200 us recording. C2 is run with and without the repeats. Parts that compare with sham cannot be
tested.

In [ ]:
DCS = [6, 30, 60]


def dc_test(ct, kind, w, reading):
    c = col(kind, "awake", w)
    d = L[(L.state == "awake") & (L.prf == 1500) & (L.celltype == ct)]
    dc_rats = set(d[d.tbd != 200].rat)
    if reading == "C1":
        d = d[d.rat.isin(dc_rats)]
    if reading == "C2 Rep2":
        d = d[~d.repeat]
    return tfus.group_test({dc: d[d.dc == dc][c].values for dc in DCS})


DC_READINGS = [(w, r) for w in ("W1", "W2") for r in ("C1", "C2 Rep1", "C2 Rep2")]
print("Recordings in the DC comparison:")
d = L[(L.state == "awake") & (L.prf == 1500)]
print(d.groupby(["dc", "rat"]).session.unique().to_string())

## fig7a-rsu-time-locked-dc

> Awake RSU time-locked responses at 1500 Hz PRF are significantly lower at 6% DC than at 60% DC
> (and, by the Figure 7A legend, than at 30% DC), and every DC is significantly higher than sham.

In [ ]:
rows = []
for w, r in DC_READINGS:
    o, post = dc_test("RSU", "tl", w, r)
    if (w, r) == ("W1", "C1"):
        show(o, post, "awake RSU 0-67 ms by DC, W1 C1")
    rows.append({"reading": f"{w} {r}", "n": o["n"], **{f"mean {k}%": round(v, 1) for k, v in o["mean"].items()},
                 "(1) KW": o["p"] < 0.05, "(2) 6 below 60": lower(post, 6, 60), "(4) 6 below 30": lower(post, 6, 30),
                 "p_bonf 6-60": tfus.pair(post, 6, 60).p_bonf, "p_bonf 6-30": tfus.pair(post, 6, 30).p_bonf})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMean by rat and DC (W1):")
print(by_rat("awake", "RSU", "tl", "W1", "Rep1", tbd=40).join(by_rat("awake", "RSU", "tl", "W1", "Rep1", tbd=400)).join(
    select("awake", "RSU", "Rep1", prf=1500).groupby("rat").tl_w1.mean().round(1).rename("30 (1500 Hz, 200 us)"), how="outer").to_string())

**Result: not reproduced.** The DC effect is significant (Kruskal-Wallis p = 1.5e-22, W1 C1),
but 6% is the highest group, not the lowest: 180.8% at 6% and 147.5% at 60% (W1), and 90.4 to 98.7% at 30%
under every reading. 6% does not differ from 60% (corrected p = 1.0) and is far above 30%. The 6%
and 60% effects come mostly from BH506 (249.0 and 209.9%); BH521 and BH526 give 113 to 121%. Part
(3) needs the sham and is not tested.

## fig7a-60-highest-time-locked-mean

> By mean values 60% DC gives the strongest RSU time-locked response, though it is only
> significantly higher than 6%.

In [ ]:
rows = []
for w, r in DC_READINGS:
    o, post = dc_test("RSU", "tl", w, r)
    rows.append({"reading": f"{w} {r}", **{f"mean {k}%": round(v, 1) for k, v in o["mean"].items()},
                 "highest": max(o["mean"], key=o["mean"].get), "p_bonf 60 vs 30": tfus.pair(post, 60, 30).p_bonf})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** The highest mean is at 6% DC under every reading (176.3 to 180.8%),
with 60% at 147.1 to 147.5%. 60% is significantly above 30% (corrected p < 1e-16), where the claim
says only 6% differs from it.

## fig7b-fsu-time-locked-dc

> FSUs show no time-locked response across DC levels; no significant differences are observed.

A claim of absence, judged on the Kruskal-Wallis across 6, 30 and 60% (G1) at 0.05 and the post
hoc tests at Bonferroni-corrected 0.05.

In [ ]:
rows = []
for w, r in DC_READINGS:
    o, post = dc_test("FSU", "tl", w, r)
    if (w, r) == ("W1", "C1"):
        show(o, post, "awake FSU 0-67 ms by DC, W1 C1")
    rows.append({"reading": f"{w} {r}", "n": o["n"], **{f"median {k}%": round(v, 1) for k, v in o["median"].items()},
                 **{f"mean {k}%": round(v, 1) for k, v in o["mean"].items()},
                 "KW p": o["p"], "significant pairs": int(post.sig.sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMean by rat and DC (W1):")
print(by_rat("awake", "FSU", "tl", "W1", "Rep1", tbd=40).join(by_rat("awake", "FSU", "tl", "W1", "Rep1", tbd=400)).to_string())

**Result: not reproduced.** FSU time-locked responses differ across DC under every reading:
Kruskal-Wallis p = 3.5e-12 (W1 C1) or smaller, with all three pairs significant. FSU respond
strongly at 6% and 60% (means 156.8 and 205.3%, medians 115.0 and 128.6%) and not at 30% (median
100.8%); the largest responses are in BH506 (338.6 and 403.3%).

## fig7c-rsu-delayed-dc

> Awake RSU delayed responses prefer 30% DC over 6% and 60%, all three DCs differ (Figure 7C
> legend), and at 60% DC they do not differ from sham.

Part (3) compares with sham and cannot be tested. The window is on bin edges (W1 = W2).

In [ ]:
rows = []
for r in ("C1", "C2 Rep1", "C2 Rep2"):
    o, post = dc_test("RSU", "del", "W1", r)
    show(o, post, f"awake RSU 0.5-1 s by DC, {r}")
    rows.append({"reading": r, "(1) KW": o["p"] < 0.05, "(2) 30 above 6 and 60": lower(post, 6, 30) and lower(post, 60, 30),
                 "(4) 6 and 60 differ": sig(post, 6, 60)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced, resting on an assumption.** With the 30% group taken from the same rats
as the 6% and 60% recordings (C1, the reading adopted), the DC effect on RSU delayed responses is
not significant (Kruskal-Wallis p = 0.21) and no pair differs. With every awake 1500 Hz, 200 us
recording as the 30% group (C2), the effect is significant (p = 2.2e-04 Rep1, 3.7e-05 Rep2) and 30%
is above 6% and 60%, but 6% and 60% do not differ (corrected p = 0.071): partially reproduced under
C2. Part (3) needs the sham and is not tested.

## fig7d-fsu-delayed-dc

> Awake FSU delayed responses are significantly higher at 60% DC than at 6%, and at 6% DC they do
> not differ from sham.

Part (3) compares with sham and cannot be tested.

In [ ]:
rows = []
for r in ("C1", "C2 Rep1", "C2 Rep2"):
    o, post = dc_test("FSU", "del", "W1", r)
    show(o, post, f"awake FSU 0.5-1 s by DC, {r}")
    rows.append({"reading": r, "(1) KW": o["p"] < 0.05, "(2) 60 above 6": lower(post, 6, 60)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under every reading the DC effect on FSU delayed responses is
significant (Kruskal-Wallis p = 0.011 C1, 4.5e-07 C2 Rep1, 1.4e-06 C2 Rep2) and 60% is above 6%
(corrected p = 0.015). Part (3), 6% against sham, needs the sham and is not tested. The difference
is small and both groups are below baseline (means 99.6% at 60% and 98.2% at 6%).

## sham-no-auditory-response (blocked)

> The off-target sham at 1500 Hz PRF produced no significant response, so the activation is not
> attributed to an auditory or skull-conduction confound.

The criterion blocks this claim (the paper names no test of the sham), and the dandiset holds no
sham recording, so the sham's responses cannot be shown either.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig3-cell-type-sorting | Partially reproduced: counts match in two recordings; the k-means features cannot be computed |
| fig4-rsu-time-locked-both-states | Blocked: data missing (no sham) |
| fig4hi-fsu-delayed-latency | Not reproduced |
| fig4h-awake-latency-wider | Not reproduced |
| fig5a-rsu-time-locked-prf-awake | Not reproduced |
| fig5c-rsu-time-locked-prf-anesth | Not reproduced |
| rsu-time-locked-anesthesia-factor | Not reproduced |
| fig5bd-fsu-time-locked-no-prf | Not reproduced |
| fig5e-rsu-delayed-prf-awake | Not reproduced |
| fig5f-fsu-delayed-prf-awake | Not reproduced |
| fig5gh-delayed-prf-anesth | Not reproduced |
| delayed-anesthesia-factor | Reproduced (assumption: Rep2) |
| fig6ad-time-adjacent-batch | Blocked: data missing (second batch) |
| fig6ef-rsu-ispta-rise-then-fall | Partially reproduced (assumption: W2) |
| fig6fh-fsu-ispta | Not reproduced |
| fig5a-1500-highest-time-locked-mean | Not reproduced |
| fig7a-rsu-time-locked-dc | Not reproduced |
| fig7a-60-highest-time-locked-mean | Not reproduced |
| fig7b-fsu-time-locked-dc | Not reproduced |
| fig7c-rsu-delayed-dc | Not reproduced (assumption: C1) |
| fig7d-fsu-delayed-dc | Reproduced |
| sham-no-auditory-response | Blocked: methods underspecified, and no sham data |

**How the analysis differs from the paper's.** Cell types are the dataset's labels; the k-means
on waveform features cannot be run, as there are no waveforms. The files hold 500 trials (the
paper's 505 less five), and five recordings hold 203 to 339. No sham recording exists, so every
comparison with sham is left out. The anesthetized PRF series has labelled units from two rats
(the paper: ten), and BH512 has no 30 Hz recording. The DC series has 6% and 60% recordings from
three rats and no 30% recording of its own. Ten awake recordings are `_2` repeats the paper does
not describe; results are shown with and without them. No unit was excluded for a fluctuating
baseline or for movement, as the dataset marks none.

**What was found about the dataset.** Twelve of the 75 files are second copies of other files:
the anesthetized recordings of BH498 and BH512 are each deposited twice (with labels under
subject folders BH497 and BH514, without labels under BH498 and BH512), BH512's 1500 Hz
anesthetized recording a third time, and one 30 Hz recording is filed under both BH498 and
BH508. The condition of a recording is coded only in its identifier; the `_2` suffix and the
2.2 s `stop_time` are not described. Fifteen anesthetized files have no cell-type labels. Every
file gives age P24W, where the paper gives 7 to 10 months, and the labelled anesthetized copies
are described as awake. The second batch of five rats, the sham condition and the 30% DC
recordings are not in the dandiset. In most 200 us recordings no time-locked population response
can be detected (10 of 64 recordings show one at t > 3, among them 4 of the 6 awake 40 and 400 us
recordings), while the 6% and 60% DC recordings respond strongly, FSU included.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")